# Three Study Architectures (Roadmap Step 8)

Implements the three GAT-based session recommendation models that form the core comparison:

| Model | Encoder | Second Phase |
|-------|---------|-------------|
| **GAT-SR-GNN** | GAT | SR-GNN soft-attention readout (local + global hybrid) |
| **GAT-TAGNN** | GAT | TAGNN target-aware attention |
| **GAT-SAGPool** | GAT | SAGPool hierarchical graph pooling |

All models share:
- The same `GATEncoder` (multi-layer, multi-head Graph Attention Network)
- The same session-graph format from `03_graph_construction.ipynb`
- Output logits of shape `[batch_size, num_items]` for `CrossEntropyLoss`

In [ ]:
from __future__ import annotations

import gzip
import pickle
from collections import Counter
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from torch_geometric.data import Batch, Data
from torch_geometric.loader import DataLoader as PyGDataLoader
from torch_geometric.nn import GATConv, SAGPooling, global_add_pool, global_max_pool
from torch_geometric.utils import softmax as pyg_softmax
from torch.utils.data import Dataset as TorchDataset

PROJECT_ROOT = Path.cwd().resolve()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

YOOCHOOSE_PATH = PROCESSED_DIR / "yoochoose_1_64_preprocessed.pkl.gz"
DIGINETICA_PATH = PROCESSED_DIR / "diginetica_preprocessed.pkl.gz"

print(f"Yoochoose processed exists: {YOOCHOOSE_PATH.exists()}")
print(f"Diginetica processed exists: {DIGINETICA_PATH.exists()}")

## Data loading and graph construction helpers

In [ ]:
Session = tuple[int, pd.Timestamp, list[int]]


def load_payload(path: Path) -> dict:
    with gzip.open(path, "rb") as f:
        return pickle.load(f)


def payload_to_sessions(split: dict) -> list[Session]:
    return [
        (sid, pd.Timestamp(date), items)
        for sid, date, items in zip(
            split["session_ids"],
            split["session_dates"],
            split["session_item_sequences"],
        )
    ]


def build_session_graph(prefix: list[int], target: int) -> Data:
    """Convert a session prefix + target into a PyG Data object."""
    item_to_local: dict[int, int] = {}
    for item in prefix:
        if item not in item_to_local:
            item_to_local[item] = len(item_to_local)

    num_nodes = len(item_to_local)
    node_ids = list(item_to_local.keys())
    sequence = [item_to_local[item] for item in prefix]

    edge_counts: Counter[tuple[int, int]] = Counter()
    for i in range(len(sequence) - 1):
        edge_counts[(sequence[i], sequence[i + 1])] += 1

    out_degree: Counter[int] = Counter()
    for (src, _), cnt in edge_counts.items():
        out_degree[src] += cnt

    sources, targets_e, weights = [], [], []
    for (src, tgt), cnt in edge_counts.items():
        sources.append(src)
        targets_e.append(tgt)
        weights.append(cnt / out_degree[src])

    edge_index = torch.tensor([sources, targets_e], dtype=torch.long)
    edge_weight = torch.tensor(weights, dtype=torch.float)

    if len(sources) == 0:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_weight = torch.zeros(0, dtype=torch.float)

    return Data(
        x=torch.tensor(node_ids, dtype=torch.long),
        edge_index=edge_index,
        edge_weight=edge_weight,
        sequence=torch.tensor(sequence, dtype=torch.long),
        last_click=torch.tensor(sequence[-1], dtype=torch.long),
        y=torch.tensor(target, dtype=torch.long),
        num_nodes=num_nodes,
    )


class SessionGraphDataset(TorchDataset):
    """Flat dataset of session-prefix graphs.

    Materialises all prefix-target pairs up front for O(1) random access.
    """

    def __init__(self, sessions: list[Session]) -> None:
        super().__init__()
        self.graphs: list[Data] = []
        for _, _, items in sessions:
            for end_idx in range(1, len(items)):
                prefix = items[:end_idx]
                target = items[end_idx]
                self.graphs.append(build_session_graph(prefix, target))

    def __len__(self) -> int:
        return len(self.graphs)

    def __getitem__(self, idx: int) -> Data:
        return self.graphs[idx]

## Load datasets and compute vocabulary size

In [ ]:
yc_payload = load_payload(YOOCHOOSE_PATH)
dg_payload = load_payload(DIGINETICA_PATH)

yc_train = payload_to_sessions(yc_payload["train"])
yc_test = payload_to_sessions(yc_payload["test"])
dg_train = payload_to_sessions(dg_payload["train"])
dg_test = payload_to_sessions(dg_payload["test"])


def get_num_items(sessions: list[Session]) -> int:
    """Max item ID + 1 (item 0 reserved as padding)."""
    return max(item for _, _, items in sessions for item in items) + 1


yc_num_items = get_num_items(yc_train + yc_test)
dg_num_items = get_num_items(dg_train + dg_test)

print(
    f"Yoochoose  — train: {len(yc_train):,}  test: {len(yc_test):,}  num_items: {yc_num_items:,}"
)
print(
    f"Diginetica — train: {len(dg_train):,}  test: {len(dg_test):,}  num_items: {dg_num_items:,}"
)

## GATEncoder

Multi-layer, multi-head GAT encoder shared by all three architectures.
Replaces the GGNN from SR-GNN / TAGNN.

In [ ]:
class GATEncoder(nn.Module):
    """Multi-layer GAT encoder with item embedding lookup.

    Parameters
    ----------
    num_items : int
        Vocabulary size (number of unique items). Item IDs in [0, num_items).
    hidden_dim : int
        Embedding and hidden representation size.
    num_layers : int
        Number of stacked GATConv layers.
    num_heads : int
        Attention heads per layer.
    dropout : float
        Dropout on both attention coefficients and features.
    concat_heads : bool
        True  -> concatenate heads (per-head dim = hidden_dim // num_heads).
        False -> average heads  (per-head dim = hidden_dim).
    residual : bool
        Add residual (skip) connection around each GATConv layer.
    """

    def __init__(
        self,
        num_items: int,
        hidden_dim: int = 100,
        num_layers: int = 1,
        num_heads: int = 4,
        dropout: float = 0.1,
        concat_heads: bool = True,
        residual: bool = True,
    ) -> None:
        super().__init__()

        if concat_heads and hidden_dim % num_heads != 0:
            raise ValueError(
                f"hidden_dim ({hidden_dim}) must be divisible by num_heads "
                f"({num_heads}) when concat_heads=True"
            )

        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.dropout = dropout
        self.concat_heads = concat_heads
        self.residual = residual

        self.embedding = nn.Embedding(num_items, hidden_dim, padding_idx=0)

        head_dim = hidden_dim // num_heads if concat_heads else hidden_dim

        self.gat_layers = nn.ModuleList()
        for _ in range(num_layers):
            self.gat_layers.append(
                GATConv(
                    in_channels=hidden_dim,
                    out_channels=head_dim,
                    heads=num_heads,
                    concat=concat_heads,
                    dropout=dropout,
                    add_self_loops=True,
                )
            )

    def forward(self, x: Tensor, edge_index: Tensor) -> Tensor:
        """Return node embeddings [total_nodes, hidden_dim] after GAT propagation."""
        h = self.embedding(x)

        for gat in self.gat_layers:
            h_in = h
            h = gat(h, edge_index)
            h = F.elu(h)
            h = F.dropout(h, p=self.dropout, training=self.training)
            if self.residual:
                h = h + h_in

        return h

## Model 1 — GAT-SR-GNN

Replaces GGNN with GAT, keeps the SR-GNN session readout:
- **Local** representation = embedding of last clicked item
- **Global** representation = soft-attention weighted sum over all session nodes
- **Hybrid** = linear projection of `[local ; global]`
- **Score** = dot product of hybrid representation with every item embedding

In [ ]:
class GATSRGNN(nn.Module):
    def __init__(
        self,
        num_items: int,
        hidden_dim: int = 100,
        num_layers: int = 1,
        num_heads: int = 4,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        self.encoder = GATEncoder(num_items, hidden_dim, num_layers, num_heads, dropout)
        self.hidden_dim = hidden_dim

        # Soft-attention parameters: alpha_i = q^T sigma(W1 h_i + W2 s_l + b)
        self.W1 = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W2 = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.q = nn.Linear(hidden_dim, 1, bias=False)

        # Hybrid projection: s_h = W3 [s_l ; s_g]
        self.W3 = nn.Linear(2 * hidden_dim, hidden_dim, bias=True)

    def forward(self, batch: Batch) -> Tensor:
        """Return logits [batch_size, num_items]."""
        h = self.encoder(batch.x, batch.edge_index)  # [N_total, d]

        # --- local: last-clicked item per graph ---
        last_local_idx = self._last_click_indices(batch)  # [N_total] mask
        s_l = h[last_local_idx]  # [B, d]

        # --- global: soft-attention readout ---
        # Expand s_l to every node in its graph
        s_l_expanded = s_l[batch.batch]  # [N_total, d]
        attn_logits = self.q(torch.sigmoid(self.W1(h) + self.W2(s_l_expanded))).squeeze(
            -1
        )  # [N_total]
        attn_weights = pyg_softmax(attn_logits, batch.batch)  # per-graph softmax
        s_g = torch.zeros_like(s_l)
        s_g.scatter_add_(
            0, batch.batch.unsqueeze(-1).expand_as(h), attn_weights.unsqueeze(-1) * h
        )

        # --- hybrid ---
        s_h = self.W3(torch.cat([s_l, s_g], dim=-1))  # [B, d]

        # --- scores: dot product with all item embeddings ---
        item_embs = self.encoder.embedding.weight  # [V, d]
        logits = s_h @ item_embs.T  # [B, V]
        return logits

    @staticmethod
    def _last_click_indices(batch: Batch) -> Tensor:
        """Global node index of last clicked item for each graph.

        Each Data object carries `last_click` — the local index of the final
        click. After batching this becomes a [B] tensor. Adding `ptr[:-1]`
        converts local to global node indices.
        """
        return batch.last_click + batch.ptr[:-1]

## Model 2 — GAT-TAGNN

Extends GAT-SR-GNN with **target-aware attention** (Yu et al. 2020):
- Compute SR-GNN-style hybrid representation
- For the target item, compute attention over session nodes conditioned on the target embedding
- Combine both representations for scoring

During training the target is the known next item. At inference, all candidates are scored.

In [ ]:
class GATTAGNN(nn.Module):
    def __init__(
        self,
        num_items: int,
        hidden_dim: int = 100,
        num_layers: int = 1,
        num_heads: int = 4,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        self.encoder = GATEncoder(num_items, hidden_dim, num_layers, num_heads, dropout)
        self.hidden_dim = hidden_dim

        # SR-GNN soft-attention parameters
        self.W1 = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W2 = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.q = nn.Linear(hidden_dim, 1, bias=False)
        self.W3 = nn.Linear(2 * hidden_dim, hidden_dim, bias=True)

        # Target-aware attention
        self.W_t = nn.Linear(hidden_dim, hidden_dim, bias=False)

    def _session_repr(self, h: Tensor, batch: Batch) -> tuple[Tensor, Tensor]:
        """SR-GNN-style hybrid representation. Returns (s_h, s_l)."""
        last_idx = GATSRGNN._last_click_indices(batch)
        s_l = h[last_idx]

        s_l_expanded = s_l[batch.batch]
        attn_logits = self.q(torch.sigmoid(self.W1(h) + self.W2(s_l_expanded))).squeeze(
            -1
        )
        attn_weights = pyg_softmax(attn_logits, batch.batch)
        s_g = torch.zeros_like(s_l)
        s_g.scatter_add_(
            0, batch.batch.unsqueeze(-1).expand_as(h), attn_weights.unsqueeze(-1) * h
        )

        s_h = self.W3(torch.cat([s_l, s_g], dim=-1))
        return s_h, s_l

    def forward(self, batch: Batch) -> Tensor:
        """Return logits [batch_size, num_items]."""
        h = self.encoder(batch.x, batch.edge_index)  # [N_total, d]
        s_h, _ = self._session_repr(h, batch)  # [B, d]

        item_embs = self.encoder.embedding.weight  # [V, d]

        # Target-aware attention: for every candidate v, compute
        # beta_i(v) = softmax_i( h_i^T W_t e_v )  over nodes in each graph
        # s_ta(v) = sum_i beta_i(v) * h_i
        #
        # Efficient computation:
        #   projected_h = h @ W_t  -> [N_total, d]
        #   raw_scores = projected_h @ item_embs.T  -> [N_total, V]
        #   beta = per-graph softmax over nodes for each candidate
        #   s_ta = scatter(beta * h, batch) -> [B, V, d]... too large.
        #
        # Instead: score = s_h @ item_embs.T + target_aware_score
        # where target_aware_score[b, v] = sum_{i in graph_b} beta_i(v) * h_i^T e_v
        #
        # Factored form avoids [B, V, d]:
        #   target_aware_score[b, v] = sum_i softmax_i(proj_h_i . e_v) * (h_i . e_v)

        proj_h = self.W_t(h)  # [N_total, d]
        # Compute raw attention logits: [N_total, V]
        raw_attn = proj_h @ item_embs.T
        # Per-graph softmax over nodes for each candidate
        beta = pyg_softmax(raw_attn, batch.batch)  # [N_total, V]
        # h_i . e_v for all nodes and candidates
        h_dot_e = h @ item_embs.T  # [N_total, V]
        # Weighted sum per graph
        ta_scores = torch.zeros(batch.num_graphs, item_embs.size(0), device=h.device)
        ta_scores.scatter_add_(
            0,
            batch.batch.unsqueeze(-1).expand_as(h_dot_e),
            beta * h_dot_e,
        )  # [B, V]

        base_scores = s_h @ item_embs.T  # [B, V]
        logits = base_scores + ta_scores
        return logits

## Model 3 — GAT-SAGPool

Fully attention-based architecture:
- **Encoder**: GAT
- **Pooling**: SAGPooling — learns to score and select important nodes
- **Readout**: concatenation of sum-pool and max-pool over retained nodes
- **Prediction**: MLP head producing scores over all items

In [ ]:
class GATSAGPool(nn.Module):
    def __init__(
        self,
        num_items: int,
        hidden_dim: int = 100,
        num_layers: int = 1,
        num_heads: int = 4,
        dropout: float = 0.1,
        pool_ratio: float = 0.5,
    ) -> None:
        super().__init__()
        self.encoder = GATEncoder(num_items, hidden_dim, num_layers, num_heads, dropout)
        self.hidden_dim = hidden_dim

        self.sagpool = SAGPooling(hidden_dim, ratio=pool_ratio, GNN=GATConv)

        # Readout: concat of sum-pool and max-pool -> 2*d
        self.fc1 = nn.Linear(2 * hidden_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, num_items)
        self.dropout = nn.Dropout(dropout)

    def forward(self, batch: Batch) -> Tensor:
        """Return logits [batch_size, num_items]."""
        h = self.encoder(batch.x, batch.edge_index)  # [N_total, d]

        # SAGPool: score nodes and keep top-k fraction
        h_pool, edge_index_pool, _, batch_pool, perm, score = self.sagpool(
            h, batch.edge_index, batch=batch.batch
        )

        # Global readout from pooled graph
        r_sum = global_add_pool(h_pool, batch_pool)  # [B, d]
        r_max = global_max_pool(h_pool, batch_pool)  # [B, d]
        r = torch.cat([r_sum, r_max], dim=-1)  # [B, 2d]

        # Prediction head
        out = F.elu(self.fc1(r))
        out = self.dropout(out)
        logits = self.fc2(out)  # [B, V]
        return logits

## Sanity check: forward pass on a small batch

Build a mini dataset from the first 5 Yoochoose sessions, run each model, verify output shapes.

In [ ]:
mini_sessions = yc_train[:5]
mini_ds = SessionGraphDataset(mini_sessions)
mini_loader = PyGDataLoader(mini_ds, batch_size=len(mini_ds), shuffle=False)
mini_batch = next(iter(mini_loader))

print(
    f"Mini batch — graphs: {mini_batch.num_graphs}, total nodes: {mini_batch.x.size(0)}"
)

NUM_ITEMS = yc_num_items
HIDDEN = 64
HEADS = 4

models = {
    "GAT-SR-GNN": GATSRGNN(NUM_ITEMS, HIDDEN, num_heads=HEADS),
    "GAT-TAGNN": GATTAGNN(NUM_ITEMS, HIDDEN, num_heads=HEADS),
    "GAT-SAGPool": GATSAGPool(NUM_ITEMS, HIDDEN, num_heads=HEADS),
}

for name, model in models.items():
    model.eval()
    with torch.no_grad():
        logits = model(mini_batch)
    expected = (mini_batch.num_graphs, NUM_ITEMS)
    assert logits.shape == expected, f"{name}: got {logits.shape}, expected {expected}"
    print(
        f"{name:15s} | output: {tuple(logits.shape)} | top-5 preds: {logits[0].topk(5).indices.tolist()}"
    )

print("\nAll forward passes OK.")

## Parameter count comparison

In [ ]:
def count_params(model: nn.Module) -> dict[str, int]:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total": total, "trainable": trainable}


rows = []
for name, model in models.items():
    c = count_params(model)
    rows.append({"model": name, **c})
    print(f"{name:15s} | total: {c['total']:>10,} | trainable: {c['trainable']:>10,}")

pd.DataFrame(rows)